In [ ]:
# Load packages
import os
import shutil
import pandas as pd 
import numpy as np 
import subprocess
from pathlib import Path

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
        "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

In [ ]:
# Remove any pre-existing PLINK2 zip
zip_file = Path("plink2_linux_x86_64_20261007.zip")
if zip_file.exists():
    zip_file.unlink()

# Download the up-to-date PLINK2 Linux 64-bit Intel release
subprocess.run(
    [
        "wget",
        "-O", str(zip_file),
        "https://s3.amazonaws.com/plink2-assets/beta1/plink2_linux_x86_64_20261019.zip"
    ],
    check=True
)

# Remove pre-existing PLINK2 files
shutil.rmtree("plink", ignore_errors=True)

# Create directory for PLINK2 files
Path("plink").mkdir(parents=True, exist_ok=True)

# Unzip PLINK2 into the plink directory
subprocess.run(
    ["unzip", "-o", str(zip_file), "-d", "plink"],
    check=True
)


In [ ]:
# Define the source and destination paths for the PLINK 2 executable
src = Path("./plink/plink2")
dst = Path("/tmp/plink2")

# Copy the PLINK 2 executable to the temporary directory
# copy2 preserves the original file metadata where possible
shutil.copy2(src, dst)

# Grant executable permissions to the copied PLINK 2 binary
dst.chmod(0o755)

# Verify that the PLINK 2 executable runs successfully
# Execute the version command and capture its output
result = subprocess.run(
    [str(dst), "--version"],
    check=True,
    capture_output=True,
    text=True
)

# Print the installed PLINK 2 version
print(result.stdout)

In [ ]:
# Define genomic regions to extract for each chromosome
# Each region is specified by chromosome, start position, and end position (bp)
regions = [
    [2, 25230000, 25320000],
    [4, 105140000, 105280000],
    [15, 90080000, 90110000],
    [17, 76730000, 76740000]
]

# Retrieve the All of Us Researcher Workbench workspace bucket
bucket = os.environ["WORKSPACE_BUCKET"]

# Iterate through the specified genomic regions
for chrom, start, stop in regions:
    
    # Define Google Cloud Storage paths for the cohort and variant inclusion lists
    cohort_gcs = (
        f"{bucket}/data/wgs_qc/analysis_samples_wgs_qc/"
        "exome/EUR-like_exome_qc.txt"
    )

    variants_gcs = (
        f"{bucket}/data/wgs_qc/wgs_variant_qc/exome/"
        f"EUR_exome_chr{chrom}_variant_qc.snplist"
    )

    # Define local filenames for downloaded input files
    cohort = "EUR-like_exome_qc.txt"
    variants = f"EUR_exome_chr{chrom}_variant_qc.snplist"

    # 1. Download cohort and variant inclusion lists from Google Cloud Storage
    # The billing project is specified to associate storage operations
    # with the appropriate Google Cloud project
    for src, dst in [(cohort_gcs, cohort), (variants_gcs, variants)]:
        subprocess.run([
            "gcloud", "storage", "cp", src, dst,
            f"--billing-project={os.environ['GOOGLE_PROJECT']}"
        ], check=True)

    print("Downloaded:", cohort, variants)

    # 2. Define the location of the chromosome-specific exome PLINK datasets
    genos='/home/dataproc/workspace/vwb-aou-datasets-controlled-v8r9/v8/wgs/short_read/snpindel/exome/plink_bed'

    # 3. Run PLINK 2 to extract the selected cohort and genomic region
    subprocess.run([
        "/tmp/plink2",
        "--bfile", f"{genos}/exome.chr{chrom}", 
        "--keep", str(cohort),
        "--extract", str(variants),
        "--chr", str(chrom),
        "--from-bp", str(start),
        "--to-bp", str(stop),
        "--force-intersect",
        "--make-bed",
        "--out", f"{os.getcwd()}/EUR_passed_chr{chrom}"
    ], check=True)

    # Define the output prefix for the generated PLINK files
    prefix = f"{os.getcwd()}/EUR_passed_chr{chrom}"

    # 4. Upload the generated PLINK binary files to the Workspace Bucket
    # Transfer the BED, BIM, and FAM files for downstream genotype-level QC
    for ext in [".bed", ".bim", ".fam"]:
        subprocess.run([
            "gcloud", "storage", "cp",
            f"{prefix}{ext}",
            f"{bucket}/data/chip/plink/",
            f"--billing-project={os.environ['GOOGLE_PROJECT']}"
        ], check=True)

In [ ]:
# Define the Ensembl gene IDs of interest for annotation extraction
gene = ['ENSG00000119772', 
        'ENSG00000168769', 
        'ENSG00000182054', 
        'ENSG00000161547']

# Retrieve the All of Us Researcher Workbench workspace bucket
bucket = os.environ["WORKSPACE_BUCKET"]

# Define a function to extract variant-level annotation information
# for the specified genes from a SAIGE-GENE+ annotation table
def extract(df, gene): 
    '''
    Extracts variant information from saige-gene+ annotation table 
    df : saige-gene+ annoation table 
    gene : list of gene of interest 
    '''
    df = df[df[0].str.contains("|".join(gene), na=False)]
    df = df[0].str.split(' ', expand=True)
    df = df.drop(columns=0).T
    df.columns = df.iloc[0]
    df = df.iloc[1:].reset_index(drop=True)
    return df 

# Initialize lists to store variant IDs and annotations
ids, annotations = [], []

# Process annotation tables for each chromosome
for chrom in [2, 4, 15, 17]:
    
    # Define GCS and local paths for the chromosome-specific annotation table
    gcs_path = (
        f"{bucket}/data/annotation_table/"
        f"EUR_HC_annotation/HC-annotation-chr{chrom}.txt"
    )
    local_path = f"HC-annotation-chr{chrom}.txt"

    # Download the annotation table from Google Cloud Storage
    subprocess.run(
        ["gcloud", "storage", "cp", gcs_path, local_path],
        check=True
    )

    # Load the annotation table and extract annotations for selected genes
    annotation = pd.read_csv(local_path, 
                             sep='\t', 
                             header=None)
    # Load the annotation table and extract annotations for selected genes
    annotation = extract(annotation, gene)
    
    # Collect variant identifiers and their corresponding annotations
    ids.append(annotation['var'])
    annotations.append(annotation)

# Concatenate variant information 
ids = np.array(pd.concat(ids))
annotations = pd.concat(annotations)


In [ ]:
# Initialize list to store variant IDs from PLINK BIM files
bims = []

# Process PLINK datasets for each chromosome
for chrom in [2, 4, 15, 17]:
   
    # Define GCS and local paths for the chromosome-specific BIM file
    gcs_path = (
        f"{bucket}/data/chip/"
        f"plink/EUR_passed_chr{chrom}.bim"
    )
    local_path = f"EUR_passed_chr{chrom}.bim"

    # Download the BIM file from Google Cloud Storage
    subprocess.run(
        ["gcloud", "storage", "cp", gcs_path, local_path],
        check=True
    )

    # Load the PLINK BIM file
    bim = pd.read_csv(local_path, 
                     sep='\t', 
                     header=None)
    
    # Collect variant identifiers from the BIM file (column 2)
    bims.append(bim[1])
    
# Combine variant IDs across all chromosomes
bims = np.array(pd.concat(bims))

# Verify that all annotated variants are present in the PLINK datasets
print(f"Check that all variants are present: {annotations['var'].isin(bims).all()}")